# Libraries and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install opencv-python-headless==4.5.5.64 tensorboardx==2.4 "torch>=2.0.0" "torchvision>=0.15.0" "protobuf~=3.20.0"

In [ ]:
import sys
sys.argv = [
    "notebook",
    "--data_name", "DUBAI",
    "--batchsize", "8",
    "--load_path", "/content/drive/MyDrive/MBRSC_Project/Submission Files/Source Code/Fine-Tuned Dubai Building Footprint Change Detection Model Best Checkpoint/Dubai Building Footprint Change Detector 2_train1__best_student_iou.pth",
]


# Evaluator

In [ ]:
import numpy as np


class Evaluator(object):
    def __init__(self, num_class):
        self.num_class = num_class
        self.confusion_matrix = np.zeros((self.num_class,)*2)

    def get_tp_fp_tn_fn(self):
        tp = np.diag(self.confusion_matrix)
        fp = self.confusion_matrix.sum(axis=0) - np.diag(self.confusion_matrix)
        fn = self.confusion_matrix.sum(axis=1) - np.diag(self.confusion_matrix)
        tn = np.diag(self.confusion_matrix).sum() - np.diag(self.confusion_matrix)
        return tp, fp, tn, fn

    def Precision(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        precision = tp / (tp + fp)
        return precision

    def Recall(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        recall = tp / (tp + fn)
        return recall

    def F1(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        Precision = tp / (tp + fp )
        Recall = tp / (tp + fn)
        F1 = (2.0 * Precision * Recall) / (Precision + Recall)
        return F1

    def OA(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        OA = (tp+tn) / (tp + fp+tn+fn)
        return OA

    def Kappa(self):
        tp, fp, tn, fn = self.get_tp_fp_tn_fn()
        PRE = ((tp + fp) * (tp + fn) + (tn + fn) * (fp + tn)) / ((tp + fp + tn + fn) * (tp + fp + tn + fn))
        OA = (tp + tn) / (tp + fp + tn + fn)
        Kappa = (OA - PRE) / (1 - PRE)
        return Kappa

    def Pixel_Accuracy(self):
        Acc = np.diag(self.confusion_matrix).sum() / self.confusion_matrix.sum()
        return Acc

    def Pixel_Accuracy_Class(self):
        Acc = np.diag(self.confusion_matrix) / self.confusion_matrix.sum(axis=1)
        Acc = np.nanmean(Acc)
        return Acc

    def Mean_Intersection_over_Union(self):
        MIoU = np.diag(self.confusion_matrix) / (
                    np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) -
                    np.diag(self.confusion_matrix))
        MIoU = np.nanmean(MIoU)
        return MIoU

    def Intersection_over_Union(self):
        IoU = np.diag(self.confusion_matrix) / (
                    np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) -
                    np.diag(self.confusion_matrix))
        return IoU

    def Frequency_Weighted_Intersection_over_Union(self):
        freq = np.sum(self.confusion_matrix, axis=1) / np.sum(self.confusion_matrix)
        iu = np.diag(self.confusion_matrix) / (
                    np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) -
                    np.diag(self.confusion_matrix))

        FWIoU = (freq[freq > 0] * iu[freq > 0]).sum()
        return FWIoU

    def _generate_matrix(self, gt_image, pre_image):
        mask = (gt_image >= 0) & (gt_image < self.num_class)
        label = self.num_class * gt_image[mask].astype('int') + pre_image[mask]
        count = np.bincount(label, minlength=self.num_class**2)
        confusion_matrix = count.reshape(self.num_class, self.num_class)
        return confusion_matrix

    def add_batch(self, gt_image, pre_image):
        assert gt_image.shape == pre_image.shape
        self.confusion_matrix += self._generate_matrix(gt_image, pre_image)

    def reset(self):
        self.confusion_matrix = np.zeros((self.num_class,) * 2)

# Data Loader

In [ ]:
import os
from PIL import Image
import torch.utils.data as data
import torchvision.transforms as transforms
import numpy as np
import random
from PIL import ImageEnhance
from PIL import ImageFilter

def safe_load_and_scale(path):
    img = Image.open(path)
    arr = np.array(img)

    # Check if the image array is 16-bit
    if arr.dtype == np.uint16 or arr.dtype == np.int16:
        # Calculate the 2nd and 98th percentiles to clip extreme bright/dark outliers
        p2, p98 = np.percentile(arr, (2, 98))

        # Clip the array to this range
        arr = np.clip(arr, p2, p98)

        # Min-max scale the clipped values to 0-255
        arr = (arr - p2) / (p98 - p2 + 1e-8) * 255.0
        arr = arr.astype(np.uint8)

        img = Image.fromarray(arr)

    return img.convert('RGB')

def cv_random_flip(img_A, img_B, label):
    # left right flip
    flip_flag = random.randint(0, 1)
    if flip_flag == 1:
        img_A = img_A.transpose(Image.FLIP_LEFT_RIGHT)
        img_B = img_B.transpose(Image.FLIP_LEFT_RIGHT)
        label = label.transpose(Image.FLIP_LEFT_RIGHT)
    return img_A, img_B, label

def cv_random_vflip(img_A, img_B, label):
    if random.randint(0, 1) == 1:
        img_A = img_A.transpose(Image.FLIP_TOP_BOTTOM)
        img_B = img_B.transpose(Image.FLIP_TOP_BOTTOM)
        label = label.transpose(Image.FLIP_TOP_BOTTOM)
    return img_A, img_B, label

def randomBlur(image_A, image_B):
    if random.random() > 0.5:
        radius = random.uniform(0.5, 1.5)
        image_A = image_A.filter(ImageFilter.GaussianBlur(radius=radius))
        image_B = image_B.filter(ImageFilter.GaussianBlur(radius=radius))
    return image_A, image_B

def randomCrop_Mosaic(image_A, image_B, label, crop_win_width, crop_win_height):
    image_width = image_A.size[0]
    image_height = image_A.size[1]
    random_region = (
        (image_width - crop_win_width) >> 1, (image_height - crop_win_height) >> 1, (image_width + crop_win_width) >> 1,
        (image_height + crop_win_height) >> 1)
    return image_A.crop(random_region), image_B.crop(random_region), label.crop(random_region)


def randomCrop(image_A, image_B, label):
    border = 30
    image_width = image_A.size[0]
    image_height = image_B.size[1]
    crop_win_width = np.random.randint(image_width - border, image_width)
    crop_win_height = np.random.randint(image_height - border, image_height)
    random_region = (
        (image_width - crop_win_width) >> 1, (image_height - crop_win_height) >> 1, (image_width + crop_win_width) >> 1,
        (image_height + crop_win_height) >> 1)
    return image_A.crop(random_region), image_B.crop(random_region), label.crop(random_region)


def randomRotation(image_A, image_B, label):
    mode = Image.BICUBIC
    if random.random() > 0.8:
        random_angle = np.random.randint(-15, 15)
        image_A = image_A.rotate(random_angle, mode)
        image_B = image_B.rotate(random_angle, mode)
        label = label.rotate(random_angle, mode)
    return image_A, image_B, label


def colorEnhance(image_A, image_B):
    bright = random.randint(5, 15) / 10.0
    contrast = random.randint(5, 15) / 10.0
    color = random.randint(0, 20) / 10.0
    sharp = random.randint(0, 30) / 10.0

    image_A = ImageEnhance.Brightness(image_A).enhance(bright)
    image_B = ImageEnhance.Brightness(image_B).enhance(bright)
    image_A = ImageEnhance.Contrast(image_A).enhance(contrast)
    image_B = ImageEnhance.Contrast(image_B).enhance(contrast)
    image_A = ImageEnhance.Color(image_A).enhance(color)
    image_B = ImageEnhance.Color(image_B).enhance(color)
    image_A = ImageEnhance.Sharpness(image_A).enhance(sharp)
    image_B = ImageEnhance.Sharpness(image_B).enhance(sharp)
    return image_A, image_B


def randomGaussian(image, mean=0.1, sigma=0.35):
    def gaussianNoisy(im, mean=mean, sigma=sigma):
        for _i in range(len(im)):
            im[_i] += random.gauss(mean, sigma)
        return im

    img = np.asarray(image)
    width, height = img.shape
    img = gaussianNoisy(img[:].flatten(), mean, sigma)
    img = img.reshape([width, height])
    return Image.fromarray(np.uint8(img))


def randomPeper(img):
    img = np.array(img)
    noiseNum = int(0.0015 * img.shape[0] * img.shape[1])
    for i in range(noiseNum):

        randX = random.randint(0, img.shape[0] - 1)

        randY = random.randint(0, img.shape[1] - 1)

        if random.randint(0, 1) == 0:

            img[randX, randY] = 0

        else:

            img[randX, randY] = 255
    return Image.fromarray(img)


# dataset for training
class ChangeDataset(data.Dataset):
    def __init__(self, root, trainsize, mosaic_ratio=0.75):
        self.trainsize = trainsize
        # get filenames
        self.image_root_A =  root + '/A/'
        self.image_root_B =  root + '/B/'
        self.gt_root = root + '/label/'
        self.mosaic_ratio = mosaic_ratio
        self.images_A = [self.image_root_A + f for f in os.listdir(self.image_root_A) if f.endswith('.jpg') or f.endswith('.png')]
        self.images_B = [self.image_root_B + f for f in os.listdir(self.image_root_B) if f.endswith('.jpg') or f.endswith('.png')]
        self.gts = [self.gt_root + f for f in os.listdir(self.gt_root) if f.endswith('.jpg')
                    or f.endswith('.png')]
        self.images_A = sorted(self.images_A)
        self.images_B = sorted(self.images_B)
        self.gts = sorted(self.gts)
        self.filter_files()
        # transforms
        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=Image.NEAREST),
            transforms.ToTensor()])
        # get size of dataset
        self.size = len(self.images_A)

    def __getitem__(self, index):
        # read imgs/gts/grads/depths
        p_ratio = random.random()
        if p_ratio > self.mosaic_ratio:
            image_A, image_B, gt = self.load_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = cv_random_vflip(image_A, image_B, gt)
            image_A, image_B, gt = randomCrop(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A, image_B = colorEnhance(image_A, image_B)
            image_A, image_B = randomBlur(image_A, image_B)
        else:
            image_A, image_B, gt = self.load_mosaic_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A,image_B = colorEnhance(image_A,image_B)

        image_A = self.img_transform(image_A)
        image_B = self.img_transform(image_B)
        gt = self.gt_transform(gt)
        gt = (gt > 0.5).float()
        return image_A, image_B, gt


    def load_img_and_mask(self, index):
        A = Image.open(self.images_A[index]).convert('RGB')
        B = Image.open(self.images_B[index]).convert('RGB')
        mask = Image.open(self.gts[index]).convert('L')
        return A, B, mask

    def load_mosaic_img_and_mask(self, index):
       indexes = [index] + [random.randint(0, self.size - 1) for _ in range(3)]
       img_a_a, img_a_b, mask_a = self.load_img_and_mask(indexes[0])
       img_b_a, img_b_b, mask_b = self.load_img_and_mask(indexes[1])
       img_c_a, img_c_b, mask_c = self.load_img_and_mask(indexes[2])
       img_d_a, img_d_b, mask_d = self.load_img_and_mask(indexes[3])

       w = self.trainsize
       h = self.trainsize

       start_x = w // 4
       strat_y = h // 4
        # The coordinates of the splice center
       offset_x = random.randint(start_x, (w - start_x))
       offset_y = random.randint(strat_y, (h - strat_y))

       crop_size_a = (offset_x, offset_y)
       crop_size_b = (w - offset_x, offset_y)
       crop_size_c = (offset_x, h - offset_y)
       crop_size_d = (w - offset_x, h - offset_y)

       croped_a_a, croped_a_b, mask_crop_a = randomCrop_Mosaic(img_a_a.copy(), img_a_b.copy(), mask_a.copy(),crop_size_a[0], crop_size_a[1])
       croped_b_a, croped_b_b, mask_crop_b = randomCrop_Mosaic(img_b_a.copy(), img_b_b.copy(), mask_b.copy(),crop_size_b[0], crop_size_b[1])
       croped_c_a, croped_c_b, mask_crop_c = randomCrop_Mosaic(img_c_a.copy(), img_c_b.copy(), mask_c.copy(),crop_size_c[0], crop_size_c[1])
       croped_d_a, croped_d_b, mask_crop_d = randomCrop_Mosaic(img_d_a.copy(), img_d_b.copy(), mask_d.copy(),crop_size_d[0], crop_size_d[1])

       croped_a_a, croped_a_b, mask_crop_a = np.array(croped_a_a), np.array(croped_a_b), np.array(mask_crop_a)
       croped_b_a, croped_b_b, mask_crop_b = np.array(croped_b_a), np.array(croped_b_b), np.array(mask_crop_b)
       croped_c_a, croped_c_b, mask_crop_c = np.array(croped_c_a), np.array(croped_c_b), np.array(mask_crop_c)
       croped_d_a, croped_d_b, mask_crop_d = np.array(croped_d_a), np.array(croped_d_b), np.array(mask_crop_d)

       top_a = np.concatenate((croped_a_a, croped_b_a), axis=1)
       bottom_a = np.concatenate((croped_c_a, croped_d_a), axis=1)
       img_a = np.concatenate((top_a, bottom_a), axis=0)

       top_b = np.concatenate((croped_a_b, croped_b_b), axis=1)
       bottom_b = np.concatenate((croped_c_b, croped_d_b), axis=1)
       img_b = np.concatenate((top_b, bottom_b), axis=0)

       top_mask = np.concatenate((mask_crop_a, mask_crop_b), axis=1)
       bottom_mask = np.concatenate((mask_crop_c, mask_crop_d), axis=1)
       mask = np.concatenate((top_mask, bottom_mask), axis=0)
       mask = np.ascontiguousarray(mask)


       img_a = np.ascontiguousarray(img_a)
       img_b = np.ascontiguousarray(img_b)

       img_a = Image.fromarray(img_a)
       img_b = Image.fromarray(img_b)
       mask = Image.fromarray(mask)

       return img_a, img_b, mask

    def filter_files(self):
        assert len(self.images_A) == len(self.gts)
        assert len(self.images_A) == len(self.images_B)
        images_A = []
        images_B = []
        gts = []
        edges = []
        for img_A_path, img_B_path, gt_path in zip(self.images_A, self.images_B, self.gts):
            img_A = Image.open(img_A_path)
            img_B = Image.open(img_B_path)
            gt = Image.open(gt_path)
            if img_A.size == img_B.size:
                if img_A.size == gt.size:
                    images_A.append(img_A_path)
                    images_B.append(img_B_path)
                    gts.append(gt_path)

        self.images_A = images_A
        self.images_B = images_B
        self.gts = gts


    def __len__(self):
        return self.size





class Test_ChangeDataset(data.Dataset):
    def __init__(self, root, trainsize):
        self.trainsize = trainsize

        image_root_A = os.path.join(root, '2019') + '/'
        image_root_B = os.path.join(root, '2023') + '/'
        gt_root = os.path.join(root, 'labels') + '/'

        # Look for TIFFs in the image folders
        self.images_A = [image_root_A + f for f in os.listdir(image_root_A) if f.endswith(('.tif', '.TIF'))]
        self.images_B = [image_root_B + f for f in os.listdir(image_root_B) if f.endswith(('.tif', '.TIF'))]
        # Look for PNGs in the labels folder
        self.gts = [gt_root + f for f in os.listdir(gt_root) if f.endswith('.png')]

        self.images_A = sorted(self.images_A)
        self.images_B = sorted(self.images_B)
        self.gts = sorted(self.gts)
        self.filter_files()

        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=Image.NEAREST),
            transforms.ToTensor()])
        self.size = len(self.images_A)

    def __getitem__(self, index):
        image_A = self.rgb_loader(self.images_A[index])
        image_B = self.rgb_loader(self.images_B[index])
        gt = self.binary_loader(self.gts[index])

        image_A = self.img_transform(image_A)
        image_B = self.img_transform(image_B)
        gt = self.gt_transform(gt)
        gt = (gt > 0.5).float()

        # Strip the 4-character .tif extension instead of .png
        file_name = self.images_A[index].split('/')[-1][:-4]

        return image_A, image_B, gt, file_name

    def filter_files(self):
        assert len(self.images_A) == len(self.gts)
        assert len(self.images_A) == len(self.images_B)
        images_A = []
        images_B = []
        gts = []
        for img_A_path, img_B_path, gt_path in zip(self.images_A, self.images_B, self.gts):
            img_A = Image.open(img_A_path)
            img_B = Image.open(img_B_path)
            gt = Image.open(gt_path)
            if img_A.size == img_B.size:
                if img_A.size == gt.size:
                    images_A.append(img_A_path)
                    images_B.append(img_B_path)
                    gts.append(gt_path)
        self.images_A = images_A
        self.images_B = images_B
        self.gts = gts

    def rgb_loader(self, path):
        # Replaced standard open with 16-bit safe loader
        return safe_load_and_scale(path)

    def binary_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('L')

    def __len__(self):
        return self.size



def get_loader(root, batchsize, trainsize, num_workers=1, shuffle=True, pin_memory=True):

    dataset =ChangeDataset(root = root, trainsize= trainsize)
    data_loader = data.DataLoader(dataset=dataset,
                                  batch_size=batchsize,
                                  shuffle=shuffle,
                                  num_workers=num_workers,
                                  pin_memory=pin_memory)
    return data_loader


def get_test_loader(root, batchsize, trainsize, num_workers=1, shuffle=True, pin_memory=True):

    dataset =Test_ChangeDataset(root = root, trainsize= trainsize)
    data_loader = data.DataLoader(dataset=dataset,
                                  batch_size=batchsize,
                                  shuffle=shuffle,
                                  num_workers=num_workers,
                                  pin_memory=pin_memory)
    return data_loader



class SemiChangeDataset(data.Dataset):
    def __init__(self, root, trainsize, train_ratio=1, mosaic_ratio=0.25):
        self.trainsize = trainsize
        self.mosaic_ratio = mosaic_ratio

        self.labeled_A = os.path.join(root, 'Labelled', 'train', '2019')
        self.labeled_B = os.path.join(root, 'Labelled', 'train', '2023')
        self.labeled_gt = os.path.join(root, 'Labelled', 'train', 'labels')

        self.unlabeled_A = os.path.join(root, 'Unlabelled', '2019')
        self.unlabeled_B = os.path.join(root, 'Unlabelled', '2023')

        # Load Labelled files (TIF for images, PNG for masks)
        self.img_A_labeled = sorted([os.path.join(self.labeled_A, f) for f in os.listdir(self.labeled_A) if f.endswith(('.tif', '.TIF'))])
        self.img_B_labeled = sorted([os.path.join(self.labeled_B, f) for f in os.listdir(self.labeled_B) if f.endswith(('.tif', '.TIF'))])
        self.gts_labeled = sorted([os.path.join(self.labeled_gt, f) for f in os.listdir(self.labeled_gt) if f.endswith('.png')])

        # Load Unlabelled files (TIF only)
        self.img_A_unlabeled = sorted([os.path.join(self.unlabeled_A, f) for f in os.listdir(self.unlabeled_A) if f.endswith(('.tif', '.TIF'))])
        self.img_B_unlabeled = sorted([os.path.join(self.unlabeled_B, f) for f in os.listdir(self.unlabeled_B) if f.endswith(('.tif', '.TIF'))])

        self.images_A = self.img_A_labeled + self.img_A_unlabeled
        self.images_B = self.img_B_labeled + self.img_B_unlabeled

        self.gts = self.gts_labeled + [None] * len(self.img_A_unlabeled)
        self.gt_list = list(range(len(self.img_A_labeled)))

        print(f'Total Training Samples: {len(self.images_A)}')
        print(f'Labelled: {len(self.img_A_labeled)}, Unlabelled: {len(self.img_A_unlabeled)}')

        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=Image.NEAREST),
            transforms.ToTensor()])

        self.size = len(self.images_A)
        # Classify each labeled tile as positive or negative
        self.positive_indices = []
        self.negative_indices = []
        for idx in self.gt_list:
            mask = Image.open(self.gts[idx]).convert('L')
            if np.array(mask).max() > 0:
                self.positive_indices.append(idx)
            else:
                self.negative_indices.append(idx)

        print(f'Positive tiles: {len(self.positive_indices)}, Negative tiles: {len(self.negative_indices)}')

    def __getitem__(self, index):
        p_ratio = random.random()
        if p_ratio > self.mosaic_ratio:
            image_A, image_B, gt = self.load_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = cv_random_vflip(image_A, image_B, gt)
            image_A, image_B, gt = randomCrop(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A, image_B = colorEnhance(image_A, image_B)
            image_A, image_B = randomBlur(image_A, image_B)
        else:
            image_A, image_B, gt = self.load_mosaic_img_and_mask(index)
            image_A, image_B, gt = cv_random_flip(image_A, image_B, gt)
            image_A, image_B, gt = randomRotation(image_A, image_B, gt)
            image_A,image_B = colorEnhance(image_A,image_B)

        image_A = self.img_transform(image_A)
        image_B = self.img_transform(image_B)
        gt = self.gt_transform(gt)
        gt = (gt > 0.5).float()
        with_label = index in self.gt_list
        return image_A, image_B, gt, with_label

    def load_img_and_mask(self, index):
        # Use the 16-bit safe loader for the satellite imagery
        A = safe_load_and_scale(self.images_A[index])
        B = safe_load_and_scale(self.images_B[index])

        if index in self.gt_list:
            mask = Image.open(self.gts[index]).convert('L')
        else:
            # Generate dummy mask for unlabelled data
            mask = Image.new('L', A.size, 0)

        return A, B, mask

    def load_mosaic_img_and_mask(self, index):
       if index in self.gt_list:
           pool = self.gt_list  # labeled index → only pull from labeled tiles
       else:
           pool = list(range(self.size))
       indexes = [index]
       indexes.append(random.choice(self.positive_indices))
       indexes += [random.choice(pool) for _ in range(2)]
       random.shuffle(indexes)
       img_a_a, img_a_b, mask_a = self.load_img_and_mask(indexes[0])
       img_b_a, img_b_b, mask_b = self.load_img_and_mask(indexes[1])
       img_c_a, img_c_b, mask_c = self.load_img_and_mask(indexes[2])
       img_d_a, img_d_b, mask_d = self.load_img_and_mask(indexes[3])

       w = self.trainsize
       h = self.trainsize

       start_x = w // 4
       strat_y = h // 4
       offset_x = random.randint(start_x, (w - start_x))
       offset_y = random.randint(strat_y, (h - strat_y))

       crop_size_a = (offset_x, offset_y)
       crop_size_b = (w - offset_x, offset_y)
       crop_size_c = (offset_x, h - offset_y)
       crop_size_d = (w - offset_x, h - offset_y)

       croped_a_a, croped_a_b, mask_crop_a = randomCrop_Mosaic(img_a_a.copy(), img_a_b.copy(), mask_a.copy(),crop_size_a[0], crop_size_a[1])
       croped_b_a, croped_b_b, mask_crop_b = randomCrop_Mosaic(img_b_a.copy(), img_b_b.copy(), mask_b.copy(),crop_size_b[0], crop_size_b[1])
       croped_c_a, croped_c_b, mask_crop_c = randomCrop_Mosaic(img_c_a.copy(), img_c_b.copy(), mask_c.copy(),crop_size_c[0], crop_size_c[1])
       croped_d_a, croped_d_b, mask_crop_d = randomCrop_Mosaic(img_d_a.copy(), img_d_b.copy(), mask_d.copy(),crop_size_d[0], crop_size_d[1])

       croped_a_a, croped_a_b, mask_crop_a = np.array(croped_a_a), np.array(croped_a_b), np.array(mask_crop_a)
       croped_b_a, croped_b_b, mask_crop_b = np.array(croped_b_a), np.array(croped_b_b), np.array(mask_crop_b)
       croped_c_a, croped_c_b, mask_crop_c = np.array(croped_c_a), np.array(croped_c_b), np.array(mask_crop_c)
       croped_d_a, croped_d_b, mask_crop_d = np.array(croped_d_a), np.array(croped_d_b), np.array(mask_crop_d)

       top_a = np.concatenate((croped_a_a, croped_b_a), axis=1)
       bottom_a = np.concatenate((croped_c_a, croped_d_a), axis=1)
       img_a = np.concatenate((top_a, bottom_a), axis=0)

       top_b = np.concatenate((croped_a_b, croped_b_b), axis=1)
       bottom_b = np.concatenate((croped_c_b, croped_d_b), axis=1)
       img_b = np.concatenate((top_b, bottom_b), axis=0)

       top_mask = np.concatenate((mask_crop_a, mask_crop_b), axis=1)
       bottom_mask = np.concatenate((mask_crop_c, mask_crop_d), axis=1)
       mask = np.concatenate((top_mask, bottom_mask), axis=0)
       mask = np.ascontiguousarray(mask)

       img_a = np.ascontiguousarray(img_a)
       img_b = np.ascontiguousarray(img_b)

       img_a = Image.fromarray(img_a)
       img_b = Image.fromarray(img_b)
       mask = Image.fromarray(mask)

       return img_a, img_b, mask

    def __len__(self):
        return self.size

def get_semiloader(root, batchsize, trainsize, train_ratio, num_workers=1, shuffle=True, pin_memory=True):
    from torch.utils.data import WeightedRandomSampler

    dataset = SemiChangeDataset(root=root, trainsize=trainsize, train_ratio=train_ratio)

    # positive tiles get higher weight, negative tiles get lower weight
    # Unlabeled tiles get same weight as negative (they're mostly negative)
    n_pos = len(dataset.positive_indices)
    n_neg = len(dataset.negative_indices)
    n_unlabeled = len(dataset.img_A_unlabeled)

    pos_set = set(dataset.positive_indices)
    neg_set = set(dataset.negative_indices)

    weights = []
    for idx in range(len(dataset)):
        if idx in pos_set:
            weights.append(1.0 / n_pos)
        elif idx in neg_set:
            weights.append(1.0 / n_neg)
        else:
            weights.append(1.0 / (n_neg + n_unlabeled))

    sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(dataset),
        replacement=True
    )

    data_loader = data.DataLoader(
        dataset=dataset,
        batch_size=batchsize,
        sampler=sampler,  # replaces shuffle
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    return data_loader

# Neural Network Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# Shared Attention Blocks
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        # Ensure ratio doesn't reduce channels below 1
        reduced = max(in_planes // ratio, 1)
        self.fc1 = nn.Conv2d(in_planes, reduced, 1, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(reduced, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(combined))


# Decoder Block: upsample, skip, refine
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super(DecoderBlock, self).__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatches from odd input dimensions
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


# Full U-Net Decoder
class UNetDecoder(nn.Module):
    def __init__(self, channel=64):
        super(UNetDecoder, self).__init__()

        # Reduce l5 to working channels
        self.conv5 = nn.Conv2d(512, channel * 4, 1)

        # Decoder path: l5 -> l4 -> l3 -> l2 -> l1
        self.dec4 = DecoderBlock(channel * 4, 512, channel * 4)  # fuse with l4
        self.dec3 = DecoderBlock(channel * 4, 256, channel * 2)  # fuse with l3
        self.dec2 = DecoderBlock(channel * 2, 128, channel)      # fuse with l2
        self.dec1 = DecoderBlock(channel, 64, channel)           # fuse with l1

        self.pred = nn.Sequential(
            nn.Conv2d(channel, channel, 3, padding=1),
            nn.BatchNorm2d(channel),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel, 1, 1)
        )

    def forward(self, l1, l2, l3, l4, l5):
        x = self.conv5(l5)         # at 1/16 resolution
        x = self.dec4(x, l4)       # 1/8
        x = self.dec3(x, l3)       # 1/4
        x = self.dec2(x, l2)       # 1/2
        x = self.dec1(x, l1)       # 1/1
        return self.pred(x)


# Main Model
class SemiModel(nn.Module):
    def __init__(self, channel=64):
        super(SemiModel, self).__init__()

        vgg16_bn = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
        self.inc = vgg16_bn.features[:5]      # -> 64 channels, 1/1 res (after first pool, actually 1/2... see note)
        self.down1 = vgg16_bn.features[5:12]  # -> 128 channels, 1/2
        self.down2 = vgg16_bn.features[12:22] # -> 256 channels, 1/4
        self.down3 = vgg16_bn.features[22:32] # -> 512 channels, 1/8
        self.down4 = vgg16_bn.features[32:42] # -> 512 channels, 1/16

        # Attention at every difference level
        self.ca1 = ChannelAttention(64);  self.sa1 = SpatialAttention()
        self.ca2 = ChannelAttention(128); self.sa2 = SpatialAttention()
        self.ca3 = ChannelAttention(256); self.sa3 = SpatialAttention()
        self.ca4 = ChannelAttention(512); self.sa4 = SpatialAttention()
        self.ca5 = ChannelAttention(512); self.sa5 = SpatialAttention()

        self.decoder = UNetDecoder(channel)

    def forward(self, A, B):
        # Backbone (siamese)
        l1A = self.inc(A);   l1B = self.inc(B)
        l2A = self.down1(l1A); l2B = self.down1(l1B)
        l3A = self.down2(l2A); l3B = self.down2(l2B)
        l4A = self.down3(l3A); l4B = self.down3(l3B)
        l5A = self.down4(l4A); l5B = self.down4(l4B)

        # Absolute difference at every level
        d1 = torch.abs(l1A - l1B)
        d2 = torch.abs(l2A - l2B)
        d3 = torch.abs(l3A - l3B)
        d4 = torch.abs(l4A - l4B)
        d5 = torch.abs(l5A - l5B)

        # Apply attention
        d1 = d1 * self.ca1(d1) * self.sa1(d1)
        d2 = d2 * self.ca2(d2) * self.sa2(d2)
        d3 = d3 * self.ca3(d3) * self.sa3(d3)
        d4 = d4 * self.ca4(d4) * self.sa4(d4)
        d5 = d5 * self.ca5(d5) * self.sa5(d5)

        # Decoder uses all 5 levels
        pred = self.decoder(d1, d2, d3, d4, d5)

        # Final upsample to input size if needed
        if pred.shape[2:] != A.shape[2:]:
            pred = F.interpolate(pred, size=A.shape[2:], mode='bilinear', align_corners=True)

        attention_map = F.interpolate(d5.mean(dim=1, keepdim=True),
                                      size=A.shape[2:], mode='bilinear', align_corners=True)

        return attention_map, pred

# Model Evaluation on the Test Set

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np

from tqdm import tqdm
from PIL import Image

import time
start=time.time()

THRESHOLD = 0.7  # Model prediction threshold, determined 0.7 as the optimal value after sweeping

def predict_with_tta(net, A, B):
    # Test-time augmentation: average predictions across 4 orientations for better predictions.
    p1 = torch.sigmoid(net(A, B)[1])
    p2 = torch.sigmoid(net(A.flip(3), B.flip(3))[1]).flip(3)       # horizontal flip
    p3 = torch.sigmoid(net(A.flip(2), B.flip(2))[1]).flip(2)       # vertical flip
    p4 = torch.sigmoid(net(A.flip(2,3), B.flip(2,3))[1]).flip(2,3) # both flips
    return (p1 + p2 + p3 + p4) / 4.0

def test(test_loader, Eva_test, save_path, net):
    print("Starting test with TTA...")

    net.train(False)
    net.eval()
    for i, (A, B, mask, filename) in enumerate(tqdm(test_loader)):
        with torch.no_grad():
            A = A.cuda()
            B = B.cuda()
            Y = mask.cuda()

            output = predict_with_tta(net, A, B)

            # Threshold
            binary = (output >= THRESHOLD).float()
            pred = binary.data.cpu().numpy().astype(int)
            target = Y.cpu().numpy()

            # Save predicted masks
            for j in range(output.shape[0]):  # fixed: was reusing 'i'
                probs_array = (torch.squeeze(binary[j])).data.cpu().numpy()
                final_mask = probs_array * 255
                final_mask = final_mask.astype(np.uint8)
                final_savepath = save_path + filename[j] + '.png'
                im = Image.fromarray(final_mask)
                im.save(final_savepath)

            Eva_test.add_batch(target, pred)
    print('target.shape', target.shape)
    print('pred.shape', pred.shape)


    IoU = Eva_test.Intersection_over_Union()
    Pre = Eva_test.Precision()
    Recall = Eva_test.Recall()
    F1 = Eva_test.F1()
    OA=Eva_test.OA()
    Kappa=Eva_test.Kappa()

    print(f'[Test] Threshold: {THRESHOLD}')
    print('[Test] F1: %.4f, Precision:%.4f, Recall: %.4f, OA: %.4f, Kappa: %.4f, IoU: %.4f' % ( F1[1],Pre[1],Recall[1],OA[1],Kappa[1],IoU[1]))
    print('F1-Score: Precision: Recall: OA: Kappa: IoU: ')
    print('{:.2f} {:.2f} {:.2f} {:.2f} {:.2f} {:.2f}\n'.format(F1[1] * 100, Pre[1] * 100, Recall[1] * 100, OA[1] * 100, Kappa[1] * 100,IoU[1] * 100))


if __name__ == '__main__':
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument('--batchsize', type=int, default=16, help='training batch size')
    parser.add_argument('--trainsize', type=int, default=256, help='training dataset size')
    parser.add_argument('--gpu_id', type=str, default='0', help='train use gpu')
    parser.add_argument('--data_name', type=str, default='DUBAI', help='dataset name')
    parser.add_argument('--model_name', type=str, default='SemiModel_noema04', help='model name')

    # Dynamic path to load the fine-tuned weights (Teacher)
    parser.add_argument('--load_path', type=str,
                        default='./_train1__best_teacher_iou.pth',
                        help='path to the specific weights you want to test')

    # Save path for the predicted PNGs
    parser.add_argument('--save_path', type=str,
                        default='./Dubai Satellite Dataset - Training splits/Test_Results/',
                        help='where to save the predicted masks')


    opt = parser.parse_args()

    # set the device for training
    if opt.gpu_id == '0':
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        print('USE GPU 0')
    elif opt.gpu_id == '1':
        os.environ["CUDA_VISIBLE_DEVICES"] = "1"
        print('USE GPU 1')

    # routing for custom Dubai test set
    if opt.data_name == 'LEVIR':
        opt.test_root = '/content/dataset/LEVIR-CD256-SPLIT/test'
    elif opt.data_name == 'DUBAI':
        # Points to the folder containing '2019', '2023', and 'labels'
        opt.test_root = '/content/drive/MyDrive/MBRSC_Project/Dubai Satellite Dataset - Training splits/Dubai Satellite Dataset - Training splits by category - expanded - 3/Labelled/test'

    # Format the save directory based on dataset and model
    opt.save_path = os.path.join(opt.save_path, opt.data_name, opt.model_name) + '/'
    if not os.path.exists(opt.save_path):
        os.makedirs(opt.save_path)

    test_loader = get_test_loader(opt.test_root, opt.batchsize, opt.trainsize, num_workers=2, shuffle=False, pin_memory=True)
    Eva_test = Evaluator(num_class=2)

    # Initialize Model
    model = SemiModel().cuda()

    # Dynamic Weight Loading Logic
    if opt.load_path is not None and os.path.isfile(opt.load_path):
        print('=> Loading model from: ', opt.load_path)
        checkpoint = torch.load(opt.load_path, map_location='cuda')

        # Handle the different dict formats saved by the training loop
        if 'best_student_net ' in checkpoint:
            print("=> Loaded Student Model (Best Epoch)")
            model.load_state_dict(checkpoint['best_student_net '], strict=False)
        elif 'net' in checkpoint:
            print("=> Loaded Student Model (Last Checkpoint)")
            model.load_state_dict(checkpoint['net'], strict=False)
        elif 'ema_net' in checkpoint:
             print("=> Loaded Teacher Model (EMA Weights)")
             model.load_state_dict(checkpoint['ema_net'], strict=False)
        else:
            print("=> Loaded Raw State Dict (e.g., best_teacher_iou.pth)")
            model.load_state_dict(checkpoint, strict=False)
    else:
        print("ERROR: Invalid weight path. Please check --load_path.")
        exit()


    test(test_loader, Eva_test, opt.save_path, model)

end=time.time()
print('The testing time of the program is:',end-start)


# Confusion Matrix

Top-Left (True Negatives): Pixels correctly predicted as "No Change".

Bottom-Right (True Positives): Pixels correctly predicted as "Change".

Top-Right (False Positives): Pixels predicted as "Change", but were actually "No Change".

Bottom-Left (False Negatives): Pixels predicted as "No Change", but were actually "Change".

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(evaluator, class_names=['No Change', 'Change']):
    matrix = evaluator.confusion_matrix

    plt.figure(figsize=(8, 6))

    sns.heatmap(matrix, annot=True, fmt='g', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)

    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

plot_confusion_matrix(Eva_test)

# Visualize samples from the test set

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

test_root = '/content/drive/MyDrive/MBRSC_Project/Submission Files/Source Code/Dubai Satellite Dataset - Training splits/Dubai Satellite Dataset - Training splits by category - expanded - 3/Labelled/test'
dir_2019 = os.path.join(test_root, '2019')
dir_2023 = os.path.join(test_root, '2023')
dir_label = os.path.join(test_root, 'labels')

pred_dir = '/content/drive/MyDrive/MBRSC_Project/Submission Files/Source Code/Model_Predictions_Tiles'

def percent_clip(array, clip_percent=2.0):
    fixed = np.zeros_like(array, dtype=np.float32)

    for i in range(3):
        band = array[:, :, i].astype(np.float32)
        p_low, p_high = np.percentile(band, (clip_percent, 100 - clip_percent))
        band = np.clip(band, p_low, p_high)

        if p_high > p_low:
            band = (band - p_low) / (p_high - p_low)
        else:
            band = band - p_low

        fixed[:, :, i] = band

    return fixed


def visualize_specific_results(file_list):
    if not file_list:
        print("No files provided in the list.")
        return

    # Filter out files that don't actually exist in the predictions directory (optional but safe)
    valid_files = [f for f in file_list if os.path.exists(os.path.join(pred_dir, f))]

    if not valid_files:
        print("No specified prediction files found.")
        return

    if len(valid_files) < len(file_list):
        print(f"Only found {len(valid_files)} out of {len(file_list)} specified files.")

    fig, axes = plt.subplots(len(valid_files), 4, figsize=(20, 5 * len(valid_files)))

    if len(valid_files) == 1:
        axes = axes.reshape(1, -1)

    for i, filename in enumerate(valid_files):
        png_filename = filename if filename.endswith('.png') else f"{filename}.png"
        tif_filename = png_filename.replace('.png', '.tif')

        img_2019_path = os.path.join(dir_2019, tif_filename)
        img_2023_path = os.path.join(dir_2023, tif_filename)
        label_path = os.path.join(dir_label, png_filename)
        pred_path = os.path.join(pred_dir, png_filename)

        try:
            img_2019 = percent_clip(np.array(Image.open(img_2019_path)))
            img_2023 = percent_clip(np.array(Image.open(img_2023_path)))
            label = Image.open(label_path)
            pred = Image.open(pred_path)

            # --- FIXED INDEXING ---
            axes[i, 0].imshow(img_2019)
            axes[i, 0].set_title(f'2019 (TIF)\n{tif_filename}')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(img_2023)
            axes[i, 1].set_title('2023 (TIF)')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(label, cmap='gray')
            axes[i, 2].set_title('Ground Truth (PNG)')
            axes[i, 2].axis('off')

            axes[i, 3].imshow(pred, cmap='gray')
            axes[i, 3].set_title('Prediction (PNG)')
            axes[i, 3].axis('off')

        except FileNotFoundError as e:
            print(f"Skipping {filename}: {e.strerror} - {e.filename}")
            continue


    plt.tight_layout()
    plt.show()

# --- Example Usage ---
# Pass the exact names of the prediction files you want to visualize
files_to_check = [
    '000000000745.png',
    '000000000758.png',
    '000000000753.png',
    '000000000435.png'
]

visualize_specific_results(files_to_check)